In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 04 · Production with ADK and Agent Engine

Notebook 03 hand-rolled the loop. ADK collapses it: declare a `MCPToolset` and an
`LlmAgent`, and ADK runs discovery, dispatch, and the tool loop for you. We run
the agent locally, then deploy it to Vertex AI Agent Engine and invoke the
deployed endpoint.

The agent is defined in `agent/agent.py` — synchronously at import time, which
Agent Engine requires for MCP toolsets.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()
# Import the agent package from this tutorial folder (the notebook's directory).
sys.path.append(os.path.abspath("."))

from agent.agent import root_agent  # builds the MCPToolset synchronously at import
print("Loaded agent:", root_agent.name)

## Run the agent locally

`AdkApp` wraps the agent and runs the full tool loop in-process. Same question
as notebook 03 — but the discover/translate/dispatch/loop is now ADK's job.

In [ ]:
from vertexai.preview.reasoning_engines import AdkApp

QUESTION = (
    "Using the BigQuery public dataset "
    "`bigquery-public-data.usa_names.usa_1910_2013`, which girls' first name was "
    "registered the most in California (state = 'CA') in the year 1995?"
)

app = AdkApp(agent=root_agent)
for event in app.stream_query(user_id="tutorial-user", message=QUESTION):
    print(event)

## Deploy to Agent Engine

`deploy/deploy_agent_engine.py` wraps the agent in `AdkApp` and calls
`agent_engines.create(...)` with `requirements` and `extra_packages=["./agent"]`.
Two gotchas it honors: the agent/toolset are defined synchronously in
`agent/agent.py`, and the `agent` package is bundled so Agent Engine can import
it server-side. Run it from this tutorial folder (`02-tutorials/mcp-integration/`).

> **Known limitation (root-caused by live testing):** the deploy succeeds and the
> deployed agent reaches Claude and emits correct tool calls, but the remote
> BigQuery MCP calls fail. ADK surfaces it as *"MCP session connection lost"*; the
> real underlying error is **HTTP 403 Forbidden** from the MCP endpoint. It is a
> 403, not a 401 — the runtime **service-account** token authenticates but the
> managed MCP server does not authorize the service-account identity (even with
> `mcp.toolUser` + BigQuery roles). The **local** run above works end-to-end with
> your end-user identity. See the "Remote MCP from inside Agent Engine" gotcha in
> the README for workarounds.

In [ ]:
# Requires STAGING_BUCKET set in .env. This builds and deploys (takes a few minutes).
# Run from this tutorial folder so the `agent` and `deploy` packages are importable.
!python -m deploy.deploy_agent_engine

## Invoke the deployed endpoint

Paste the resource name printed by the deploy step, then query the remote agent.

In [ ]:
import vertexai
from vertexai import agent_engines

# The deployed resource lives in the Agent Engine region (AGENT_ENGINE_LOCATION),
# which is a concrete region — not the "global" Claude endpoint.
vertexai.init(
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ.get("AGENT_ENGINE_LOCATION", "us-central1"),
)

RESOURCE_NAME = "PASTE_RESOURCE_NAME_FROM_DEPLOY_STEP"
remote = agent_engines.get(RESOURCE_NAME)
for event in remote.stream_query(user_id="tutorial-user", message=QUESTION):
    print(event)

## Recap — what ADK collapsed

The entire manual loop from notebook 03 (discover → translate → dispatch →
loop, in `run_bridge`) reduces to two declarations:

```python
bigquery_mcp = MCPToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=os.environ["BQ_MCP_ENDPOINT"],
        httpx_client_factory=adc_authenticated_client,  # injects a fresh ADC token
    ),
)
root_agent = LlmAgent(model=os.environ["CLAUDE_MODEL_ID"], name="bigquery_analyst",
                      instruction=..., tools=[bigquery_mcp])
```

ADK runs the loop the bridge spelled out by hand — and the same definition
deploys to Agent Engine. Vertex still has no native connector; ADK is just a
well-factored implementation of the bridge.

## Cleanup

Agent Engine deployments are billable for as long as they exist. Delete the
resource when you are done. (Run this once you have finished experimenting.)

In [ ]:
# Delete the deployed Agent Engine so it stops incurring cost.
remote.delete(force=True)
print("Deleted Agent Engine:", RESOURCE_NAME)

# The staging bucket and the runtime service-account IAM bindings persist beyond
# this notebook. Remove them too when you are completely done, e.g.:
#   !gcloud storage rm -r {os.environ["STAGING_BUCKET"]}
#   SA="service-PROJECT_NUMBER@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
#   for role in roles/mcp.toolUser roles/bigquery.jobUser roles/bigquery.dataViewer; do
#     !gcloud projects remove-iam-policy-binding {os.environ["GOOGLE_CLOUD_PROJECT"]} \
#        --member="serviceAccount:$SA" --role="$role" --condition=None
#   done